# AML EXOME BURDEN ANALYSIS 
This script calculates rare-variant polygenic risk scores at both the gene level and the chromosomal exome level.

In [ ]:
# Load packages 
import os

In [ ]:
# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

# COMMAND 

In [ ]:
%%writefile gene_burden_analysis.sh
#!/bin/bash

# Exit on any error or unset variable
set -o errexit
set -o nounset

# Make unmatched globs expand to nothing 
shopt -s nullglob

# Loop through populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # Check that cohort file exists
    if [ ! -f "${cohort}/${pop}-like_exome_qc.txt" ]; then
        echo "Cohort file for ${pop} not found!"
        exit 1
    fi
    
    # Prepare PLINK2 binary
    cp "${plink2}" plink2
    chmod +x plink2

    echo "Number of samples in ${pop}:"
    wc -l "${cohort}/${pop}-like_exome_qc.txt"
    
    # === GENE BURDEN ANALYSIS ===
    echo "Running gene-level burden PRS for chromosome ${chrom}..."

    for gene_burden in ${gene_weights}/HC-burden-prs-*-chr${chrom}.txt; do
        echo "Processing ${gene_burden} ...."

        awk -F '\t' '{print $3}' "${gene_burden}" > "gene_${chrom}.snplist.txt"

        ./plink2 \
            --bfile "${genos}/exome.chr${chrom}" \
            --keep "${cohort}/${pop}-like_exome_qc.txt" \
            --extract "gene_${chrom}.snplist.txt" \
            --memory 3100 \
            --max-maf 0.01 \
            --mac 1 \
            --score "${gene_burden}" 3 5 6 no-mean-imputation cols=+nallele,+scoresums \
            --out "${gene_output}/${pop}_$(basename "${gene_burden}" .txt)" \
            || { 
            echo "$(basename "${gene_burden}")" >> "${gene_output}/failed_genes.txt";
            echo "PLINK failed for $(basename "${gene_burden}") — skipping.";
        }
    done

    # === EXOME BURDEN ANALYSIS ===
    echo "Running exome-wide burden PRS for chromosome ${chrom}..."
    awk -F '\t' '{print $3}' "${exome_weights}/Exome-burden-prs-chr${chrom}.txt" > exome.snplist.txt

    ./plink2 \
        --bfile "${genos}/exome.chr${chrom}" \
        --keep "${cohort}/${pop}-like_exome_qc.txt" \
        --extract exome.snplist.txt \
        --memory 3100 \
        --max-maf 0.01 \
        --mac 1 \
        --score "${exome_weights}/Exome-burden-prs-chr${chrom}.txt" 3 5 6 no-mean-imputation cols=+nallele,+scoresums \
        --out "${exome_output}/${pop}-Exome-burden-prs-chr${chrom}"

done


In [ ]:
# Copy gene_burden_analysis.sh script to my_bucket 
!gsutil cp gene_burden_analysis.sh {my_bucket}/data/scripts/

In [ ]:
# Copy gene_burden_analysis.sh script to my_bucket 
!gsutil cat {my_bucket}/data/scripts/gene_burden_analysis.sh

# SUBMIT COMMAND 

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=21
UPPER=22

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 50 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{1..2}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=1
UPPER=2

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 420 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{2..3}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=2
UPPER=3

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 300 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{2..3}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=3
UPPER=4

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 250 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{4..5}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=4
UPPER=5

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 175 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{5..7}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=5
UPPER=8

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 220 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{8..10}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=8
UPPER=11

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 175 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{11..12}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=11
UPPER=13

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 250 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{13..14}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=13
UPPER=14

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 100 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{14..18}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=14
UPPER=16

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 175 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{16..18}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=16
UPPER=17

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 200 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{17..18}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=17
UPPER=18

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 240 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{18..19}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=18
UPPER=19

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 100 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{19..20}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=19
UPPER=20

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 275 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{19..20}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=20
UPPER=21

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 120 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
# Check the path for plink files version 8 
!gsutil -u $GOOGLE_PROJECT ls -lha gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{21..22}.bed  

In [ ]:
%%bash --out burden_analysis

#############################################################################
# DSUB: SUBMIT COMMAND 
#############################################################################

# Source the dsub setup script, which contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=22
UPPER=23

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --disk-size 100 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores/" \
  --input-recursive gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}/" \
   --output-recursive exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/" \
  --output-recursive gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome" \
  --env exome_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_exome_burden_scores" \
  --env gene_weights="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_scores/EUR_HC_burden_scores_chr${chromo}" \
  --env exome_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores" \
  --env gene_output="${WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chromo}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/gene_burden_analysis.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done